# Digital Payments Settlement Latency & Merchant Churn Analytics
### Group 03 — Programming for Analytics · Comprehensive Group Capstone Project & Live Code Defense
> **TEMPLATE NOTEBOOK** — analysis scaffold: every block states the *goal*, the *hints* (tables, columns,
> formulas) and `TODO` markers for the code you must complete. Keep this notebook beside
> `payments_analytics.db` and run *Restart Kernel → Run All Cells*; it executes end-to-end once all TODOs
> are implemented. (The fully executed `..._SOLUTION.ipynb` demonstrates one valid completion.)

| # | Team Member | SAP ID | Roll Number | Email ID | % Contribution |
|---|-------------|--------|-------------|----------|----------------|
| 1 | *Member A (add name)* | *SAP ID* | *Roll No* | *email@institution.edu* | 100% |
| 2 | *Member B (add name)* | *SAP ID* | *Roll No* | *email@institution.edu* | 100% |
| 3 | *Member C (add name)* | *SAP ID* | *Roll No* | *email@institution.edu* | 100% |
| 4 | *Member D (add name)* | *SAP ID* | *Roll No* | *email@institution.edu* | 100% |
| 5 | *Member E (add name)* | *SAP ID* | *Roll No* | *email@institution.edu* | 100% |
| 6 | *Member F (add name)* | *SAP ID* | *Roll No* | *email@institution.edu* | 100% |
| 7 | *Member G (add name)* | *SAP ID* | *Roll No* | *email@institution.edu* | 100% |

**Data:** `payments_analytics.db` (SQLite 3) — 3 dimensions (`dim_merchants`, `dim_issuing_banks`,
`dim_payment_methods`) → 2 facts (`fact_payment_transactions`, `fact_merchant_settlements`).
Python ≥ 3.13 with `pandas`, `sqlalchemy`, `matplotlib` (see `requirements.txt`); relative paths only.

## 1 · Setup & Configuration *(worked example)*
Imports, display/plot settings, a read-only SQL helper that pushes aggregation down to SQLite,
and formatting utilities. Extend as needed.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")          # keep outputs clean for evaluation
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (9.5, 4.2), "axes.grid": True,
    "grid.alpha": 0.30, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "font.size": 9.5,
})

# --- database connection (RELATIVE path — the .db must sit beside this notebook) ---
DB_PATH = Path("payments_analytics.db")
assert DB_PATH.exists(), "payments_analytics.db not found — keep it beside this notebook."
engine = create_engine(f"sqlite:///{DB_PATH}")

TIER_ORDER = ["ENTERPRISE", "MID_MARKET", "SMB"]


def run_sql(sql: str) -> pd.DataFrame:
    """Run a read-only SQL query against the analytics DB (aggregation pushed down to SQLite)."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


def label_bars(ax, fmt="{:,.0f}"):
    """Value labels on top of vertical bars."""
    for p in ax.patches:
        ax.annotate(fmt.format(p.get_height()),
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=8, xytext=(0, 2), textcoords="offset points")


print("Environment ready — pandas", pd.__version__, "| DB:", DB_PATH.name)

Environment ready — pandas 3.0.5 | DB: payments_analytics.db


## 2 · Relational Schema Inspection & Integrity Validation
**Goal:** inventory the 5 tables with row counts; verify PK uniqueness, FK orphan-ness (4 relationships),
the settlement ledger identity (`net = gross − MDR`), and join fan-out (1:N facts bridging N:M dimensions).
**Hints:** `sqlite_master` lists tables; `PRAGMA table_info`/`PRAGMA foreign_key_list` give PKs/FKs;
orphan check = `LEFT JOIN parent ... WHERE parent.pk IS NULL`.
**Deliverable:** assertions (orphan/dup counts == 0) + a written verdict on the schema shape.

In [2]:
# Worked example — table inventory with row counts:
inventory = run_sql("SELECT name, type FROM sqlite_master WHERE type = 'table' ORDER BY name")
for t in inventory["name"]:
    n = run_sql(f"SELECT COUNT(*) AS n FROM {t}")["n"].iloc[0]
    print(f"{t:<30s} {n:>9,} rows")

# TODO: declare the 4 FK relationships and prove zero orphans
#       (fact_payment_transactions -> dim_merchants / dim_issuing_banks / dim_payment_methods,
#        fact_merchant_settlements -> dim_merchants)  — assert count == 0 for each.
# TODO: prove PK uniqueness on all 5 tables (GROUP BY pk HAVING COUNT(*) > 1 must return nothing).
# TODO: validate the settlement ledger identity:
#       ABS(net_settled_amount_inr - (gross_volume_inr - net_mdr_deducted_inr)) <= 0.01 for every row.
# TODO: report join fan-out (max/avg txns and settlements per merchant; distinct banks/methods per
#       merchant via the facts) and write a 2-3 line verdict on 1:N vs N:M-through-facts.
print("TODO: PK / FK-orphan / ledger-identity / fan-out checks.")

dim_issuing_banks                     12 rows
dim_merchants                      8,500 rows
dim_payment_methods                   16 rows
fact_merchant_settlements        125,000 rows
fact_payment_transactions        530,000 rows
TODO: PK / FK-orphan / ledger-identity / fan-out checks.


## 3 · Data Quality Audit & Treatment Decisions
**Goal:** NULL audit across all tables; categorical domain conformance (`status`, `error_code`,
`sla_breach_flag`, tiers, churn logic); date-range coverage; value-plausibility bounds. Then a
**documented decisions table** — every material cleaning/treatment choice justified by a check.
**Hints:** only `churn_date`/`churn_reason` should be NULL (actives); `SUCCESS ⇔ ERR_NONE` must pair
1:1; settlement dates spill into a 2025-01 stub month (T+2/T+3 lag) — decide how monthly series handle it;
settlement coverage is partial by merchant — check whether coverage selects on churn.
**Deliverable:** checks table with zero violations + markdown decisions table (Q1…Q6).

In [3]:
# TODO: NULL audit — for every column of every table, count NULLs; explain any structurally
#       meaningful NULLs (churn fields of active merchants).
# TODO: domain checks with counts of violations (target: all zero):
#       status in {SUCCESS, FAILED, USER_DROPPED, TIMEOUT};
#       error_code in the 6-value dictionary domain;
#       (status = 'SUCCESS') == (error_code = 'ERR_NONE');
#       sla_breach_flag in {0,1}; settlement_delay_days >= 0;
#       latency_ms within 250-12000; amount_inr > 0; tier domain; churn consistency (is_active vs churn_date).
# TODO: temporal coverage table: min/max + distinct months for transactions, settlement batches,
#       actual settlement dates, merchant onboarding; identify any stub months beyond 2024-12.
print("TODO: NULL + domain + temporal coverage checks.")

TODO: NULL + domain + temporal coverage checks.


In [4]:
# TODO: summarise your data-quality findings & treatment decisions in a markdown table
#       (use IPython.display.Markdown). One row per decision: Finding -> Decision & rationale.
#       Cover at least: churn NULLs, ledger/domain cleanliness, the 2025-01 settlement stub month,
#       latency bounds, and the settlement-coverage population.
print("TODO: decisions table (Markdown).")

TODO: decisions table (Markdown).


## 4 · Analytical Preparation (push-down to SQL)
**Goal:** define the shared success-rate expression (`AVG(CASE WHEN status='SUCCESS' THEN 1.0 ELSE 0 END)*100`),
a reusable merchant-level breach-exposure CTE over `fact_merchant_settlements`
(batches, breach_rate, gpv, mdr + exposure buckets `0% / 0-10% / 10-25% / >25%`), and quantify settlement
coverage by tier (covered vs not-covered merchants and churn rates).
**Hints:** push aggregation into SQL; Python should only receive small result frames; the CTE is reused
by KBQ 4 and KBQ 8 — keep one single definition.
**Deliverable:** coverage pivot + a note on whether partial coverage biases churn comparisons.

In [5]:
# Worked example — the success-rate expression used everywhere:
SR = "AVG(CASE WHEN t.status = 'SUCCESS' THEN 1.0 ELSE 0 END) * 100"

# TODO: define MERCHANT_BREACH_CTE (per-merchant batches, breach_rate, gpv_inr, mdr_inr,
#       merchant_tier, is_churned, churn_reason, breach_bucket) so downstream blocks reuse it.
# TODO: coverage analysis — merchants with/without settlements by tier: counts + churn rates;
#       conclude whether exposure->churn analysis within the covered population is unbiased.
print("TODO: MERCHANT_BREACH_CTE + coverage analysis.")

TODO: MERCHANT_BREACH_CTE + coverage analysis.


## 5 · Descriptive Baseline — the Business at a Glance
**Goal:** one KPI band (transactions, success rate, avg latency, attempted GMV, settlement batches,
SLA breach rate, realized MDR, merchants, churn rate) + monthly volume/SR/latency trend + channel mix
(successful GMV, txn count, potential MDR by channel).
**Hints:** 18-month window 2023-07..2024-12; remember UPI MDR = 0%.
**Deliverable:** KPI table, `channel` DataFrame, a two-panel chart (monthly volume bars + SR line;
channel GMV bars) with title/labels/units, and a 3-4 line baseline read-out.

In [6]:
# TODO: KPI query — one row joining fact-level aggregates (use SR from Block 4) + scalar subqueries
#       for settlements, breach rate, MDR, merchants, churn rate.
# TODO: monthly = volume, SR%, avg latency per month (SUBSTR(txn_timestamp, 1, 7)).
# TODO: channel  = successful GMV, txns, potential MDR (amount * mdr_rate_pct / 100) per channel.
print("TODO: KPI band + monthly trend + channel mix.")

TODO: KPI band + monthly trend + channel mix.


In [7]:
# TODO: two-panel figure:
#   left  — monthly volume bars (000s) + success-rate line on a twin axis (legend combined);
#   right — successful GMV by channel with value labels.
# TODO: print() a baseline read-out referencing the computed numbers (no hardcoded figures).
print("TODO: baseline charts + read-out.")

TODO: baseline charts + read-out.


## 6 · Part 1 — KBQ 1: Issuing-Bank Latency & Switch-Degradation Profiling
**Goal:** per-bank volume, success rate, timeout %, avg latency, **p95**, spike share (latency ≥ 10,000 ms);
PSU vs private rollup; intraday latency/spike profile.
**Hints:** SQLite has no PERCENTILE_CONT — use `ROW_NUMBER() OVER (PARTITION BY bank ORDER BY latency_ms)`
and take rank `CAST(0.95*n AS INT)+1`; hour = `CAST(SUBSTR(txn_timestamp,12,2) AS INT)`;
weight tier rollups by txns (pandas `np.average(..., weights=...)`).
**Deliverable:** bank profile table, tier comparison, two-panel chart (avg latency by bank with p95 labels;
hourly spikes bars + avg-latency line), findings on worst banks and the evening window.

In [8]:
# TODO: bank_profile — bank_name, tier, txns, SR%, timeout%, avg latency, spike% (>= 10,000 ms), p95.
# TODO: tier_cmp — txns-weighted rollup for TIER_1_PSU / TIER_1_PVT / TIER_2_PVT.
print("TODO: bank profile + PSU-vs-private rollup.")

TODO: bank profile + PSU-vs-private rollup.


In [9]:
# TODO: hourly profile — txns, SR%, avg latency, spike count per hour of day.
# TODO: two-panel figure: (left) avg latency by bank (horizontal bars, colored by tier, p95 labels);
#       (right) hourly spike bars + avg-latency line — quantify the 18:00-22:00 evening window.
# TODO: print() findings: PSU vs private gap, worst three banks, evening vs off-peak latency & spikes.
print("TODO: gateway charts + findings.")

TODO: gateway charts + findings.


## 7 · Part 1 — KBQ 2: Error-Code Pareto — Infrastructure vs User-Side
**Goal:** Pareto of `error_code` over all non-SUCCESS attempts with cumulative %; split failures into
Infrastructure (`ERR_BANK_TIMEOUT`, `ERR_SWITCH_UNAVAILABLE`, `ERR_NPCI_DEGRADED`) vs User-side
(`ERR_INSUFFICIENT_FUNDS`, `ERR_AUTH_FAILED`); check the `USER_DROPPED ⇔ ERR_AUTH_FAILED` pairing.
**Hints (accounting discipline):** `status='TIMEOUT'` rows and `error_code='ERR_BANK_TIMEOUT'` rows are
different counts — never sum them.
**Deliverable:** Pareto DataFrame (share %, cumulative %), pairing check, Pareto chart
(bars + cumulative line on twin axis), infrastructure-vs-user split with interpretation.

In [10]:
# TODO: pareto — error_code, fault_domain (Infrastructure/User-side), failures, share_pct, cumulative_pct.
# TODO: pairing — (status, error_code) counts for non-SUCCESS; confirm USER_DROPPED 1:1 ERR_AUTH_FAILED.
# TODO: counts for the accounting note — status='TIMEOUT' rows vs ERR_BANK_TIMEOUT-on-FAILED rows.
print("TODO: error Pareto + pairing checks.")

TODO: error Pareto + pairing checks.


In [11]:
# TODO: Pareto chart — failure bars colored by fault domain, cumulative % line on twin axis,
#       title carrying total failures and % of all attempts.
# TODO: print() the infrastructure vs user-side split and the two accounting notes.
print("TODO: Pareto chart + read-out.")

TODO: Pareto chart + read-out.


## 8 · Part 2 — KBQ 3: Settlement Delay Diagnostics by Contracted SLA Cycle
**Goal:** breach rate and delay severity per contracted cycle (`dim_merchants.settlement_cycle_sla`,
T_PLUS_0..T_PLUS_3); delay-day distribution; monthly breach trend (cut at 2024-12); **MCC × cycle
breach matrix (multivariate diagnostic)**.
**Hints:** `sla_breach_flag` is 0/1; `AVG(CASE WHEN sla_breach_flag=1 THEN settlement_delay_days END)`
gives delay-if-breach; batch month = `SUBSTR(settlement_batch_date,1,7)`.
**Deliverable:** cycle table, delay distribution, monthly trend with min–max range, matrix pivot +
heatmap, and the T+0/T+1 vs T+2/T+3 structural-gap finding.

In [12]:
# TODO: cycle — batches, breach %, avg delay days, avg delay-if-breach per contracted cycle.
# TODO: delay_dist — batch counts per settlement_delay_days (0..4+).
# TODO: monthly_breach — batches + breach % per batch month (WHERE settlement_batch_date <= '2024-12-31');
#       print the min-max range (seasonality check).
print("TODO: settlement diagnostics.")

TODO: settlement diagnostics.


In [13]:
# TODO: mcc_matrix — breach % pivot mcc_category x settlement_cycle_sla; display it.
# TODO: three-panel figure: breach % by cycle (bars) · delay-day distribution (bars) · MCC x cycle
#       heatmap (imshow + annotated cells, colorbar).
# TODO: print() the structural-gap finding (T+0/T+1 ~12% vs T+2/T+3 ~50%; breached batches ~1.5 days late).
print("TODO: MCC x cycle matrix + charts + finding.")

TODO: MCC x cycle matrix + charts + finding.


## 9 · Part 2 — KBQ 4: Settlement Friction vs Merchant Attrition *(multivariate diagnostic)*
**Goal:** churn rate by tier; **churn rate by breach-exposure bucket × tier** (the mandatory
multi-factor diagnostic); churn-reason decomposition (focus SMB); churned-merchant tenure.
**Hints:** reuse `MERCHANT_BREACH_CTE` from Block 4; buckets `0% / 0-10% / 10-25% / >25%`;
tenure = `JULIANDAY(churn_date) - JULIANDAY(onboarding_date)`.
**Deliverable:** churn pivot (+ counts), SMB ratio (>25% bucket vs 0-10% bucket), grouped-bar chart,
SMB churn-reason bar, and the mechanism statement tying breaches to attrition.

In [14]:
# TODO: churn_tier — merchants, churned, churn % per tier.
# TODO: churn_matrix — merchants / churned / churn % per (merchant_tier, breach_bucket) via the CTE;
#       pivot to tier x bucket (order buckets 0% / 0-10% / 10-25% / >25%).
# TODO: print() the SMB exposure gradient and the MID_MARKET/ENTERPRISE comparison.
print("TODO: churn x exposure x tier cross-tab.")

TODO: churn x exposure x tier cross-tab.


In [15]:
# TODO: reasons — churned merchants per (merchant_tier, churn_reason); isolate SMB.
# TODO: tenure — avg days on platform per tier for churned merchants.
# TODO: two-panel figure: churn rate by breach bucket x tier (grouped bars) · SMB churn reasons (bars).
# TODO: print() the mechanism: % of SMB churn citing SETTLEMENT_DELAY + average churned tenure.
print("TODO: churn charts + mechanism statement.")

TODO: churn charts + mechanism statement.


## 10 · Part 3 — KBQ 5 & 6: Dynamic SLA Smart-Routing — A/B Lift & Latency Mitigation
**Goal:** A/B success rate for `is_routed_via_dynamic_sla = 0` vs `1`; verify the **benchmark:
+8.50 pp absolute lift (accepted band 8.30–8.70 pp)**; lift stability by tier and by channel;
latency mitigation (avg, p50, p95 by route; route × bank).
**Hints:** p50/p95 via the same ROW_NUMBER trick; keep the A/B split as the *only* difference in the
comparison; assert the overall lift lands inside the band.
**Deliverable:** A/B table with `assert 8.30 <= lift <= 8.70`, stability pivots, latency-distribution
histogram + worst-bank route comparison chart, quantified mitigation findings.

In [16]:
# TODO: ab — txns, SR%, avg latency for routed 0 vs 1; compute lift = SR(1) - SR(0).
# TODO: lift_tier / lift_channel — SR pivots with lift_pp column by tier and by channel.
# print(f"SUCCESS-RATE LIFT: standard ... -> dynamic ... = +... pp")
# assert 8.30 <= lift <= 8.70, "Routing lift outside accepted benchmark band 8.30-8.70 pp!"   # benchmark
print("TODO: A/B lift + benchmark assertion + stability pivots.")

TODO: A/B lift + benchmark assertion + stability pivots.


In [17]:
# TODO: latency_route — avg, p50, p95 latency per route; compute reduction in ms and %.
# TODO: route_bank — avg latency pivot (bank x route) + improvement_ms; display the worst six switches.
# TODO: figures — overlaid latency histograms by route; worst-6-bank standard vs dynamic bars.
# TODO: print() mitigation findings (avg -x ms / -x%, p95 improvement, clawback on degraded PSU switches).
print("TODO: latency mitigation analysis + charts.")

TODO: latency mitigation analysis + charts.


## 11 · Part 4 — KBQ 7: MDR Revenue Economics by Tier × Channel
**Goal:** two accounting lenses — **realized** MDR from `fact_merchant_settlements`
(`net_mdr_deducted_inr` vs `gross_volume_inr` by tier) and **potential** MDR from successful
transactions × `dim_payment_methods.mdr_rate_pct` (by channel and tier × channel); monthly realized
MDR trend (cut at 2024-12).
**Hints:** keep realized vs potential distinct; blended % = MDR/GPV; UPI = 0%.
**Deliverable:** realized-by-tier, potential-by-channel (with GMV share and rate range), tier×channel
pivot, monthly trend, three-panel chart, and the UPI-volume-vs-card-revenue read-out.

In [18]:
# TODO: realized_tier — batches, GPV, realized MDR, blended % per merchant tier.
# TODO: potential_channel — successful GMV, GMV share %, potential MDR, rate range per channel.
# TODO: tier_channel — potential MDR pivot tier x channel.
# TODO: monthly_mdr — realized MDR per settlement month (actual_settlement_date <= '2024-12-31').
print("TODO: realized + potential MDR economics.")

TODO: realized + potential MDR economics.


In [19]:
# TODO: three-panel figure — tier GPV + MDR (twin axes) · potential MDR by channel with GMV-share
#       labels · monthly realized MDR line.
# TODO: print() read-outs: SMB share of realized MDR; UPI GMV share at 0% MDR; credit-card share of
#       potential MDR from ~20% of GMV.
print("TODO: economics charts + read-out.")

TODO: economics charts + read-out.


## 12 · Part 4 — KBQ 8: Protected Volume — What Eliminating Settlement Delays Saves
**Goal:** size the at-risk SMB segment (> 25% of batches breaching) and the annualized GMV + MDR
protected by fixing breaches. Two scenarios: **at-stake ceiling** (whole segment run-rate) and
**excess-churn scenario** (segment churn reverts to the low-breach SMB baseline; protected share =
excess pp).
**Hints:** reuse `MERCHANT_BREACH_CTE`; annualize trailing 18-month run-rate × 12/18 (justify with the
flat monthly breach/MDR trends you produced); state every assumption.
**Deliverable:** scenario table (merchants, churn rates, excess pp, saveable merchants, annualized
GMV/MDR ceiling, protected GMV/MDR), bar chart of both scenarios, printed assumption list.

In [20]:
# TODO: at_risk — merchants, churned, churn %, 18-mo GPV/MDR for SMB with breach_bucket = '>25%';
#       annualize: x 12/18.
# TODO: baseline — SMB churn % in the '0-10%' bucket; excess_pp = at-risk churn - baseline.
#       saveable merchants = n_risk * excess_pp/100; protected GPV/MDR = segment annualized x excess_pp/100.
# TODO: scenario table (metric / value rows) + two-scenario bar chart (GMV left axis, MDR right axis).
# TODO: print() the four assumptions (run-rate annualization, churn reversion, average economics,
#       no replacement-acquisition credit).
print("TODO: protected-volume scenarios.")

TODO: protected-volume scenarios.


## 13 · Findings → 5 Prioritized Recommendations & Executive Summary
**Goal:** convert evidence into **5 prioritized recommendations**, each as
**Finding → Business Problem → Recommendation → Expected Impact**, impact quantified from *this
notebook's own outputs* (never hardcoded) with assumptions stated; flag the top 3 for the deck; close
with an executive summary table.
**Hints:** build the markdown with `IPython.display.Markdown` and f-strings referencing computed
variables; keep the recommendation chain explicit; impacts in revenue/retention terms.

In [21]:
# from IPython.display import Markdown
# TODO: exec_md = Markdown(f\"\"\"
#   ### Recommendations (prioritized)
#   Rec 1 ★ (settlement fix for breach-exposed SMB) — Finding / Problem / Recommendation / Expected impact
#   Rec 2 ★ (complete smart-routing rollout + PSU de-congestion) — ...
#   Rec 3 ★ (save-desk at the 20-25% breach inflection) — ...
#   Rec 4   (UPI monetization via value-added rails) — ...
#   Rec 5   (evening switch SLOs) — ...
#   ### Executive summary  (evidence table: scale / gateway / settlement / churn / prize)
#   \"\"\" — every number interpolated from a computed variable; assumptions inline.
# ipy_display(exec_md)
print("TODO: 5 recommendations + executive summary.")

TODO: 5 recommendations + executive summary.
